In [1]:
import pandas as pd
import nltk
import string
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords as nltk_stopwords
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import jaccard_score
import torch
from sentence_transformers import SentenceTransformer

In [2]:
df = pd.read_excel('Job Name Data.xlsx')

In [3]:
df.head()

,JP_ID,JobTitle,Description
0,1204211,General Manager,Preferred: General Manager experience in limi...
1,1206798,Project Manager,Strategic Leadership: To clearly articula...
2,1211880,Area Manager,Mission 1: Management Manager as a role mode...
3,1206619,Deputy Program Manager (Nutritionist),Job Descriptions / Responsibilities: Under th...
4,1209527,Nutrition & Entrepreneurship Capacity building...,The Nutrition & Entrepreneurship Capacity buil...


In [4]:
df.shape

(8048, 3)

In [5]:
df.duplicated().sum()

0

In [6]:
stopwords = stopwords.words('english')
wn = WordNetLemmatizer()

In [7]:
stop_words = set(nltk_stopwords.words('english'))

In [8]:
def preprocess_text(text):
    # Tokenization
    tokens = word_tokenize(text)

    # Stopword removal
    filtered_tokens = [word for word in tokens if word.lower() not in stop_words]

    # Lemmatization
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [lemmatizer.lemmatize(word) for word in filtered_tokens]

    # Join the tokens back into a single string
    processed_text = ' '.join(lemmatized_tokens)

    return processed_text

In [9]:
df['Description'] = df['Description'].apply(preprocess_text)

In [10]:
new_df = df[['JP_ID', 'Description']]

print(new_df)

        JP_ID                                        Description
0     1204211  Preferred : General Manager experience limited...
1     1206798  Strategic Leadership : clearly articulate cont...
2     1211880  Mission 1 : Management Manager role model : em...
3     1206619  Job Descriptions / Responsibilities : direct g...
4     1209527  Nutrition & Entrepreneurship Capacity building...
...       ...                                                ...
8043  1207204  Arrive destination schedule . Fulfill administ...
8044  1206760  Enhancing student ` ability fluent confident E...
8045  1205843  Elaborating client product . Communicate regul...
8046  1217520  Conduct Bangla class V-VI level student . Prep...
8047  1204402  Monitor type sample process Check sample accor...

[8048 rows x 2 columns]


In [11]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(df['Description'])

# Calculate cosine similarity
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print("Cosine Similarity:")
print(cosine_sim)

Cosine Similarity:
[[1.         0.14250602 0.1074986  ... 0.02615562 0.03626233 0.04429514]
 [0.14250602 1.         0.17003966 ... 0.04944585 0.07436993 0.02615275]
 [0.1074986  0.17003966 1.         ... 0.08648044 0.03519237 0.01963813]
 ...
 [0.02615562 0.04944585 0.08648044 ... 1.         0.         0.02458325]
 [0.03626233 0.07436993 0.03519237 ... 0.         1.         0.        ]
 [0.04429514 0.02615275 0.01963813 ... 0.02458325 0.         1.        ]]


In [12]:
# Calculate Jaccard similarity (convert TF-IDF matrix to binary matrix for Jaccard similarity)
binary_matrix = (tfidf_matrix > 0).astype(int)
jaccard_sim = jaccard_score(binary_matrix, binary_matrix, average=None)

print("\nJaccard Similarity:")
print(jaccard_sim)


Jaccard Similarity:
[1. 1. 1. ... 1. 1. 1.]


In [13]:
# Given dummy description
dummy_description = "Python php ML"

# Preprocessing function with lemmatization
def preprocess_text(text):
    lemmatizer = WordNetLemmatizer()
    tokens = word_tokenize(text)
    filtered_tokens = [word.lower() for word in tokens if word.lower() not in stop_words]
    lemmatized_tokens = [lemmatizer.lemmatize(word) for word in filtered_tokens]
    return ' '.join(lemmatized_tokens)

# Preprocess the dummy description
processed_dummy_description = preprocess_text(dummy_description)

# Preprocess descriptions in the DataFrame
df['Processed_Description'] = df['Description'].apply(preprocess_text)

In [14]:
df

,JP_ID,JobTitle,Description,Processed_Description
0,1204211,General Manager,Preferred : General Manager experience limited...,preferred : general manager experience limited...
1,1206798,Project Manager,Strategic Leadership : clearly articulate cont...,strategic leadership : clearly articulate cont...
2,1211880,Area Manager,Mission 1 : Management Manager role model : em...,mission 1 : management manager role model : em...
3,1206619,Deputy Program Manager (Nutritionist),Job Descriptions / Responsibilities : direct g...,job description / responsibility : direct guid...
4,1209527,Nutrition & Entrepreneurship Capacity building...,Nutrition & Entrepreneurship Capacity building...,nutrition & entrepreneurship capacity building...
...,...,...,...,...
8043,1207204,Driver,Arrive destination schedule . Fulfill administ...,arrive destination schedule . fulfill administ...
8044,1206760,English Teacher (Spoken English),Enhancing student ` ability fluent confident E...,enhancing student ` ability fluent confident e...
8045,1205843,Junior Sales Engineer,Elaborating client product . Communicate regul...,elaborating client product . communicate regul...
8046,1217520,Bangla Teacher - V-VI,Conduct Bangla class V-VI level student . Prep...,conduct bangla class v-vi level student . prep...


In [15]:
# Preprocess the dummy description
processed_dummy_description = preprocess_text(dummy_description)

# Fit TF-IDF vectorizer on processed descriptions
tfidf_matrix = tfidf_vectorizer.fit_transform(df['Processed_Description'])

# Transform dummy description using TF-IDF vectorizer
dummy_description_vector = tfidf_vectorizer.transform([processed_dummy_description])

In [16]:
# Calculate Jaccard similarity between dummy description and all real descriptions
jaccard_similarities = []
for index, row in df.iterrows():
    description_vector = tfidf_vectorizer.transform([row['Processed_Description']])
    dummy_binary_vector = (dummy_description_vector > 0).astype(int).toarray()[0]
    description_binary_vector = (description_vector > 0).astype(int).toarray()[0]
    jaccard_sim = jaccard_score(dummy_binary_vector, description_binary_vector) * 100  # Convert to percentage
    jaccard_similarities.append(jaccard_sim)

# Add scores to DataFrame
df['Score'] = jaccard_similarities

# Select top 10 job postings with highest scores
top_10_jobs = df.nlargest(10, 'Score')

print(top_10_jobs)

        JP_ID                                JobTitle  \
8030  1210672                       Junior Programmer   
5945  1214212       Web Developer & CRM Administrator   
5386  1212698    Full Stack Backend Software Engineer   
7924  1223140                       Intern Programmer   
7666  1206877  Sr. Software Engineer ( Oracle -Apex )   
7667  1206876  Sr. Software Engineer ( Oracle + PHP )   
7688  1216624              Web Developer (Full Stack)   
7714  1213842                                  Intern   
3440  1211749               System Administrator (IT)   
4146  1213648  Laravel Developer/ Assistant Developer   

                                            Description  \
8030  Expertise Red Hat 3scale , Openshift Container...   
5945  Experience Web Developer , portfolio past proj...   
5386  Requirements collection client , analysis & sy...   
7924  passionate driven individual looking kickstart...   
7666  work oracle developer 10g , 11.g , Fusion Midd...   
7667  work oracle 

In [17]:
df['Matched'] = df['Score'].apply(lambda x: 1 if x > 0 else 0)

In [18]:
# Calculate cosine similarity between dummy description and all real descriptions
cosine_similarities = cosine_similarity(dummy_description_vector, tfidf_matrix)

# Add scores to DataFrame and convert to percentage
df['Score'] = cosine_similarities[0] * 100

# Select top 10 job postings with highest scores
top_10_jobs = df.nlargest(10, 'Score')

print(top_10_jobs)

        JP_ID                                           JobTitle  \
3606  1208550                             Trainee Data Scientist   
7648  1211630              Python Developer, FastAPI (Mid-Level)   
2748  1211962           Software Engineer (Full Stack Developer)   
4289  1206141  Sr. Executive (Software Engineer PHP/Android F...   
8030  1210672                                  Junior Programmer   
7143  1214633                                WordPress Developer   
3207  1205363                                    Trainee Officer   
7714  1213842                                             Intern   
5945  1214212                  Web Developer & CRM Administrator   
7099  1209186                  Sr. PHP Developer - Web Developer   

                                            Description  \
3606  Python workability must clear understanding OO...   
7648  Key Responsibilities : Design implement robust...   
2748  least Two year ` professional experience worki...   
4289  Design , 

In [19]:
df['Matched'] = df['Score'].apply(lambda x: 1 if x > 0 else 0)

In [20]:
model = SentenceTransformer('paraphrase-MiniLM-L6-v2')

processed_dummy_description = preprocess_text(dummy_description)

vectorizer = CountVectorizer()
description_tokens = vectorizer.fit_transform(df['Processed_Description']).toarray()

# Tokenize dummy description
dummy_description_tokens = vectorizer.transform([processed_dummy_description]).toarray()[0]

# Get BERT embeddings for descriptions
description_embeddings = model.encode(df['Processed_Description'].tolist(), convert_to_tensor=True)

# Get BERT embeddings for dummy description
dummy_description_embedding = model.encode([processed_dummy_description], convert_to_tensor=True)

# Calculate cosine similarity between dummy description and all real descriptions using BERT embeddings
cosine_similarities_bert = torch.nn.functional.cosine_similarity(dummy_description_embedding, description_embeddings, dim=1)

# Calculate Jaccard similarity between dummy description and all real descriptions
similarities_jaccard = []
for tokens in description_tokens:
    similarity = len(set(dummy_description_tokens) & set(tokens)) / len(set(dummy_description_tokens) | set(tokens))
    similarities_jaccard.append(similarity)

threshold = 0.5  # Adjust threshold as needed
predictions_bert = [1 if score > threshold else 0 for score in cosine_similarities_bert]
predictions_jaccard = [1 if score > threshold else 0 for score in similarities_jaccard]

# Compute classification report for cosine similarity using BERT embeddings
print("Classification Report for Cosine Similarity (BERT Embeddings):")
print(classification_report(df['Matched'], predictions_bert))

# Compute accuracy for Cosine Similarity using BERT Embeddings
accuracy_cosine_bert = accuracy_score(df['Matched'], predictions_bert)
print("Accuracy for Cosine Similarity (BERT Embeddings):", accuracy_cosine_bert)

# Compute accuracy for Jaccard Similarity
accuracy_jaccard = accuracy_score(df['Matched'], predictions_jaccard)
print("Accuracy for Jaccard Similarity:", accuracy_jaccard)

Classification Report for Cosine Similarity (BERT Embeddings):
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      7948
           1       0.00      0.00      0.00       100

    accuracy                           0.99      8048
   macro avg       0.49      0.50      0.50      8048
weighted avg       0.98      0.99      0.98      8048

Accuracy for Cosine Similarity (BERT Embeddings): 0.9875745526838966
Accuracy for Jaccard Similarity: 0.9076789264413518


c:\Users\Pias\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Pias\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Pias\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modif

In [21]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Calculate precision, recall, and F1-score for cosine similarity using BERT embeddings
precision_cosine_bert = precision_score(df['Matched'], predictions_bert)
recall_cosine_bert = recall_score(df['Matched'], predictions_bert)
f1_cosine_bert = f1_score(df['Matched'], predictions_bert)

print("Precision for Cosine Similarity (BERT Embeddings):", precision_cosine_bert)
print("Recall for Cosine Similarity (BERT Embeddings):", recall_cosine_bert)
print("F1-score for Cosine Similarity (BERT Embeddings):", f1_cosine_bert)

# Calculate precision, recall, and F1-score for Jaccard similarity
precision_jaccard = precision_score(df['Matched'], predictions_jaccard)
recall_jaccard = recall_score(df['Matched'], predictions_jaccard)
f1_jaccard = f1_score(df['Matched'], predictions_jaccard)

print("Precision for Jaccard Similarity:", precision_jaccard)
print("Recall for Jaccard Similarity:", recall_jaccard)
print("F1-score for Jaccard Similarity:", f1_jaccard)


Precision for Cosine Similarity (BERT Embeddings): 0.0
Recall for Cosine Similarity (BERT Embeddings): 0.0
F1-score for Cosine Similarity (BERT Embeddings): 0.0
Precision for Jaccard Similarity: 0.01943198804185351
Recall for Jaccard Similarity: 0.13
F1-score for Jaccard Similarity: 0.033810143042912875


c:\Users\Pias\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [22]:
# Given dummy description
dummy_description = "Skilled in Python, PHP, and machine learning techniques, with a passion for data analysis. Also data manipulation and visualization, combined with expertise in machine learning algorithms also i know data analysis and web sceaping"

# Preprocessing function
def preprocess_text(text):
    return set(word_tokenize(text.lower()))  # Tokenize text and convert to lowercase

# Preprocess dummy description
processed_dummy_description = preprocess_text(dummy_description)

# Calculate Jaccard similarity between dummy description and all real descriptions
similarities = []
for desc in df['Processed_Description']:
    processed_desc = preprocess_text(desc)
    similarity = len(processed_dummy_description.intersection(processed_desc)) / len(processed_dummy_description.union(processed_desc))
    similarities.append(similarity)

# Convert similarities to binary predictions (1 for match, 0 for no match)
threshold = 0.0  # Adjust threshold as needed
predictions = [1 if sim > threshold else 0 for sim in similarities]

# Compute accuracy
accuracy = sum(predictions) / len(predictions)
print("Accuracy:", accuracy)

# Calculate precision, recall, and F1-score
true_labels = [1] * len(df)  # Assuming all descriptions match
precision = precision_score(true_labels, predictions)
recall = recall_score(true_labels, predictions)
f1 = f1_score(true_labels, predictions)

print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)


Accuracy: 0.9917992047713717
Precision: 1.0
Recall: 0.9917992047713717
F1-score: 0.9958827199001872


In [23]:
# Given dummy description
dummy_description = "Skilled in Python, PHP, and machine learning techniques, with a passion for data analysis"

# Preprocessing function
def preprocess_text(text):
    return set(word_tokenize(text.lower()))  # Tokenize text and convert to lowercase

# Preprocess dummy description
processed_dummy_description = preprocess_text(dummy_description)

similarities = []
for desc in df['Processed_Description']:
    processed_desc = preprocess_text(desc)
    similarity = len(processed_dummy_description.intersection(processed_desc)) / len(processed_dummy_description.union(processed_desc))
    similarities.append(similarity)

# Convert similarities to binary predictions (1 for match, 0 for no match)
threshold = 0.0  # Adjust threshold as needed
predictions = [1 if sim > threshold else 0 for sim in similarities]

# Compute accuracy
accuracy = sum(predictions) / len(predictions)
print("Accuracy:", accuracy)

# Calculate precision, recall, and F1-score
true_labels = [1] * len(df)  # Assuming all descriptions match
precision = precision_score(true_labels, predictions)
recall = recall_score(true_labels, predictions)
f1 = f1_score(true_labels, predictions)

print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

Accuracy: 0.9335238568588469
Precision: 1.0
Recall: 0.9335238568588469
F1-score: 0.965619176145492
